<a href="https://colab.research.google.com/github/10dimensions/gnc-toolbox/blob/main/davenport_q_method.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial.transform import Rotation as R

In [2]:
def davenport_q_method(reference_vectors, body_vectors, weights):
    """
    Computes the optimal attitude quaternion using Davenport's q-method.
    Solves Wahba's problem by finding the eigenvector corresponding to the
    maximum eigenvalue of the Davenport K-matrix.

    Inputs:
        reference_vectors : Nx3 array of reference vectors (e.g., in ECI)
        body_vectors      : Nx3 array of measured vectors (e.g., in Body frame)
        weights           : N array of scalar weights (sum of weights does not need to be 1)

    Returns:
        q_optimal : 1D array [x, y, z, w] (SciPy convention, w is scalar)
    """
    # =========================================================================
    # Step 1: Compute the Attitude Profile Matrix (B)
    # =========================================================================
    # B = sum( w_i * b_i * r_i^T )
    B = np.zeros((3, 3))
    for i in range(len(weights)):
        b = body_vectors[i]
        r = reference_vectors[i]
        w = weights[i]
        B += w * np.outer(b, r)

    # =========================================================================
    # Step 2: Construct Davenport's K-Matrix
    # =========================================================================
    # Calculate the sub-components of the K-matrix
    S = B + B.T                          # Symmetric part
    sigma = np.trace(B)                  # Trace of B
    # Vector part of the antisymmetric component of B
    z = np.array([
        B[1, 2] - B[2, 1],
        B[2, 0] - B[0, 2],
        B[0, 1] - B[1, 0]
    ])

    # Assemble the 4x4 K-matrix
    # Layout: Top-left 3x3 is vector part, Bottom-right 1x1 is scalar part
    K = np.zeros((4, 4))
    K[0:3, 0:3] = S - sigma * np.eye(3)
    K[0:3, 3]   = z
    K[3, 0:3]   = z
    K[3, 3]     = sigma

    # =========================================================================
    # Step 3: Eigendecomposition and Optimal Quaternion Extraction
    # =========================================================================
    # np.linalg.eigh is used because K is a real symmetric matrix.
    # It returns eigenvalues in ascending order.
    eigenvalues, eigenvectors = np.linalg.eigh(K)

    # The optimal quaternion is the eigenvector corresponding to the MAX eigenvalue
    max_idx = np.argmax(eigenvalues)
    q_optimal = eigenvectors[:, max_idx]

    # Normalize the quaternion to ensure unit magnitude
    q_optimal = q_optimal / np.linalg.norm(q_optimal)

    # Enforce the "positive scalar" convention (Double Cover resolution)
    # This ensures q and -q are treated consistently, keeping w >= 0
    if q_optimal[3] < 0:
        q_optimal = -q_optimal

    return q_optimal


In [3]:
def calculate_quaternion_error(q_true, q_est):
    """Calculates the angular error between two quaternions in degrees."""
    # Create SciPy Rotation objects
    rot_true = R.from_quat(q_true)
    rot_est = R.from_quat(q_est)

    # Relative rotation
    rot_diff = rot_true.inv() * rot_est

    # Convert to rotation vector and get magnitude (angle)
    rot_vec = rot_diff.as_rotvec()
    error_rad = np.linalg.norm(rot_vec)

    return np.degrees(error_rad)

In [4]:
# =========================================================================
# MAIN EXECUTION: Test Scenario with Multiple Sensors
# =========================================================================
if __name__ == "__main__":
    print("--- Davenport's q-method Test & Validation ---\n")

    # 1. Define a "True" Attitude
    # Let's use a random rotation for a realistic test
    true_euler = [45.0, -20.0, 15.0] # Roll, Pitch, Yaw in degrees
    rot_true = R.from_euler('XYZ', true_euler, degrees=True)
    C_true = rot_true.as_matrix()
    q_true = rot_true.as_quat() # [x, y, z, w]

    # 2. Define Reference Vectors (in ECI) and simulate Body Measurements
    # We will use 3 sensors: Sun (accurate), Mag (noisy), Nadir (medium)
    ref_sun  = np.array([1.0, 0.0, 0.0])
    ref_mag  = np.array([0.2, 0.8, 0.1])
    ref_nadir= np.array([0.0, 0.0, -1.0])

    # Generate "perfect" body measurements
    b_sun_perfect  = C_true @ ref_sun
    b_mag_perfect  = C_true @ ref_mag
    b_nadir_perfect= C_true @ ref_nadir

    # Add realistic sensor noise
    np.random.seed(42)
    # Sun sensor is very accurate (0.05 deg noise)
    noise_sun = np.random.normal(0, np.radians(0.05), 3)
    # Magnetometer is noisy (1.5 deg noise)
    noise_mag = np.random.normal(0, np.radians(1.5), 3)
    # Nadir sensor is medium (0.2 deg noise)
    noise_nadir = np.random.normal(0, np.radians(0.2), 3)

    b_sun_meas  = b_sun_perfect + noise_sun
    b_mag_meas  = b_mag_perfect + noise_mag
    b_nadir_meas= b_nadir_perfect + noise_nadir

    # 3. Define Weights (Inverse of variance roughly)
    # We trust the Sun the most, Nadir second, Mag the least.
    weights = np.array([10.0, 1.0, 5.0])

    ref_vecs = np.array([ref_sun, ref_mag, ref_nadir])
    body_vecs = np.array([b_sun_meas, b_mag_meas, b_nadir_meas])

    # 4. Run Davenport's q-method
    q_est = davenport_q_method(ref_vecs, body_vecs, weights)

    # 5. Analyze Results
    error_deg = calculate_quaternion_error(q_true, q_est)

    print(f"True Attitude (Euler): {true_euler}")
    print(f"Total Attitude Error:  {error_deg:.4f} degrees\n")

    print("True Quaternion [x, y, z, w]:")
    print(np.round(q_true, 4))

    print("\nEstimated Quaternion [x, y, z, w]:")
    print(np.round(q_est, 4))

    # Bonus: Show what happens if we give the noisy Magnetometer too much weight
    print("\n--- Sensitivity Test ---")
    bad_weights = np.array([1.0, 10.0, 1.0]) # Overweight the noisy Mag sensor
    q_bad = davenport_q_method(ref_vecs, body_vecs, bad_weights)
    bad_error = calculate_quaternion_error(q_true, q_bad)
    print(f"Error with optimal weights: {error_deg:.4f} deg")
    print(f"Error with bad weights:     {bad_error:.4f} deg")
    print("-> Davenport correctly trusts the accurate sensors more!")

--- Davenport's q-method Test & Validation ---

True Attitude (Euler): [45.0, -20.0, 15.0]
Total Attitude Error:  97.4912 degrees

True Quaternion [x, y, z, w]:
[ 0.3527 -0.2082  0.0529  0.9107]

Estimated Quaternion [x, y, z, w]:
[-0.3521  0.2084 -0.0512  0.911 ]

--- Sensitivity Test ---
Error with optimal weights: 97.4912 deg
Error with bad weights:     96.6033 deg
-> Davenport correctly trusts the accurate sensors more!
